Example of how to design and implement a simple LLM powered chatbot which can have a conversation with the user and remember the previous interactions. 

We are only using the language model for the conversation. Not any external source of information or a chatbot that takes actions. 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x126b944f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x126b94400>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

In [4]:
## A simple demonstation invoking the model with a HumanMessage to initiate a conversation. The model will respond with an AIMessage.
model.invoke([HumanMessage(content="I am Gaurav Ghosh. I am a devloper. I am learning about LLMs and LangChain.")])

AIMessage(content='Nice to meet you, Gaurav Ghosh. Learning about Large Language Models (LLMs) and LangChain can be an exciting and challenging journey.\n\nLangChain is an open-source Python library that allows developers to build complex conversational AI applications using LLMs. It provides a set of tools and interfaces for tasks such as prompt engineering, knowledge retrieval, and reasoning.\n\nTo get started with LangChain, you may want to explore the following topics:\n\n1. **Understanding LLMs**: Start by learning about the basics of LLMs, including their architecture, training process, and applications.\n2. **LangChain architecture**: Familiarize yourself with the LangChain architecture, including its components and interfaces.\n3. **Prompt engineering**: Learn about prompt engineering and how to design effective prompts for LLMs.\n4. **Knowledge retrieval**: Understand how to use LangChain for knowledge retrieval and reasoning tasks.\n5. **Integration with other libraries**: Ex

In [5]:
## Simple flow test to see if it remembers recent information in the conversation. We are giving it a system message to set the context and then asking it to recall the information we shared in the previous message.
model.invoke([SystemMessage(content="You are a helpful assistant who is going to help a developer with learning LLM and Langchain."),
              HumanMessage(content="I am Gaurav Ghosh. I am a devloper. I am learning about LLMs and LangChain."),
             HumanMessage(content="Can you tell me my name and what I do? ")])

AIMessage(content="Nice to meet you, Gaurav Ghosh! You are a developer who is currently learning about Large Language Models (LLMs) and LangChain. I'm here to help you with any questions or topics you'd like to discuss related to these subjects. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 98, 'total_tokens': 158, 'completion_time': 0.076031718, 'completion_tokens_details': None, 'prompt_time': 0.005568052, 'prompt_tokens_details': None, 'queue_time': 0.044971148, 'total_time': 0.08159977}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce103-0a53-74e1-849d-cc2a6f44339f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 60, 'total_tokens': 158})

## Message History
from mailbox import Message


We can use a MessageHistory to keep track of the conversation history and pass it to the model for generating responses. This allows the model to have context about the previous interactions and generate more coherent and relevant responses.

Also we can create and store them in a database to pass onto a chain so as to load them as part of the input. 


In [8]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [14]:
## Initializing a chat history to keep track of the conversation. 
# This will allow the model to have context of previous interactions and maintain a coherent conversation.
## Each user will have their own session history, which will be stored in a dictionary with the session ID as the key.

session_histories = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in session_histories:
        session_histories[session_id] = ChatMessageHistory()
    return session_histories[session_id]

with_message_history = RunnableWithMessageHistory(model,get_session_history)

In [22]:
config={"configurable": {"session_id": "chat_user_3"}}
session_histories

{'chat_user_1': InMemoryChatMessageHistory(messages=[SystemMessage(content='You are a software development tool assistant who is going to help a developer with learning LLM and Langchain.', additional_kwargs={}, response_metadata={}), HumanMessage(content='I am Gaurav Ghosh. I am a devloper. I am learning about LLMs and LangChain.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Can you tell me my name and what I do? ', additional_kwargs={}, response_metadata={}), AIMessage(content="Nice to meet you, Gaurav Ghosh. You are a developer, and you're currently learning about LLMs (Large Language Models) and LangChain. I'm here to assist you in your learning journey. What specific aspects of LLMs and LangChain would you like to explore further?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 100, 'total_tokens': 163, 'completion_time': 0.081929833, 'completion_tokens_details': None, 'prompt_time': 0.005560381, 'prompt_t

In [21]:
response=with_message_history.invoke(
    [SystemMessage(content="You are a software development tool assistant who is going to help a developer with learning LLM and Langchain."),
    HumanMessage(content="Can you tell me my name and what I do? ")], config=config)
response.content

'Since I don\'t have any prior information about you, I\'ll make an educated guess.\n\nLet\'s say your name is "Alex," and you are a software developer. You\'re interested in learning about LLM (Large Language Model) and Langchain, and you\'re seeking my assistance to explore these topics.\n\nFeel free to correct me if my guess is incorrect! What do you think?'

## Prompt Templates

prompt templates help to turn raw user information into a format that the LLM can work with. Here we are working a with a meassage

In [39]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant in the following language: {language}"),
    MessagesPlaceholder(variable_name="messages"),
])

chain = prompt|model

In [27]:
chain.invoke({"messages":[HumanMessage(content="I am Gaurav Ghosh. I am a devloper. I am learning about LLMs and LangChain.")], "language":"Spanish"})

AIMessage(content='¡Hola Gaurav Ghosh! Me alegra ayudarte a aprender sobre LLMs (Modelos de Lenguaje de Largo Recurrente) y LangChain. ¿Qué nivel de conocimiento tienes actualmente sobre estos temas? ¿Estás empezando desde cero o ya tienes alguna experiencia previa?\n\nSi te gustaría, podemos comenzar desde el principio y explorar juntos los conceptos básicos de LLMs, como su arquitectura, tipos de modelos y aplicaciones comunes. Luego, podemos pasar a LangChain, que es una biblioteca de Python para trabajar con LLMs y crear aplicaciones de inteligencia artificial.\n\n¿Qué te gustaría aprender primero? ¿Quieres conocer más sobre:\n\n* La arquitectura de LLMs\n* Tipos de modelos LLMs (transformadores, RNN, etc.)\n* Aplicaciones comunes de LLMs (traducción, clasificación de textos, etc.)\n* LangChain y cómo funciona\n* Ejemplos de código con LangChain\n\nEstoy aquí para ayudarte, así que no dudes en preguntar cualquier cosa que te surja. ¡Vamos a empezar!', additional_kwargs={}, response

In [28]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")

In [44]:
config = {"configurable": {"session_id": "chat_user_11"}}
response = with_message_history.invoke(
    {"messages":[
    HumanMessage(content="I am a devloper")], "language":"Bengali"}, config=config)

In [45]:
response.content

'আপনি একজন ডেভেলপার। আমি আপনাকে ল্যাংগুইজ মডেল (LLM) এবং ল্যাঙ্গচেইনের সাথে কাজ করার জন্য সহায়তা করব। কী কোন বিষয়ে আপনার কাছে প্রশ্ন আছে তা জিজ্ঞাসা করতে দিন।\n\nযদি আমি আপনাকে ল্যাংগুইজ মডেল (LLM) এবং ল্যাঙ্গচেইন সম্পর্কে শিখতে সাহায্য করতে পারি, তাহলে কিই করতে আপনি আগ্রহী?\n\n1. LLM এর মৌলিক ধারণা শিখতে\n2. Langchain সার্ভিসটে কাজ করতে\n3. LLM এবং Langchain এর মধ্যে সম্পর্ক জানতে\n4. অন্য কিছু \n\nআমার কাছে থাকা তথ্য থেকে আপনি যে কোন বিষয়ে জিজ্ঞাসা করতে পারেন।'

In [46]:
## Need to check this. LLMs are no where mentioned in 
session_histories['chat_user_11'].messages

[HumanMessage(content='I am a devloper', additional_kwargs={}, response_metadata={}),
 AIMessage(content='আপনি একজন ডেভেলপার। আমি আপনাকে ল্যাংগুইজ মডেল (LLM) এবং ল্যাঙ্গচেইনের সাথে কাজ করার জন্য সহায়তা করব। কী কোন বিষয়ে আপনার কাছে প্রশ্ন আছে তা জিজ্ঞাসা করতে দিন।\n\nযদি আমি আপনাকে ল্যাংগুইজ মডেল (LLM) এবং ল্যাঙ্গচেইন সম্পর্কে শিখতে সাহায্য করতে পারি, তাহলে কিই করতে আপনি আগ্রহী?\n\n1. LLM এর মৌলিক ধারণা শিখতে\n2. Langchain সার্ভিসটে কাজ করতে\n3. LLM এবং Langchain এর মধ্যে সম্পর্ক জানতে\n4. অন্য কিছু \n\nআমার কাছে থাকা তথ্য থেকে আপনি যে কোন বিষয়ে জিজ্ঞাসা করতে পারেন।', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 505, 'prompt_tokens': 68, 'total_tokens': 573, 'completion_time': 0.653323352, 'completion_tokens_details': None, 'prompt_time': 0.00492552, 'prompt_tokens_details': None, 'queue_time': 0.04578324, 'total_time': 0.658248872}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reaso